# **Procesamiento y Consolidación de Datos: Poesía y Métricas Musicales**

Tras el Diagnóstico Exploratorio Inicial (EDA), consolidamos las fuentes de datos literarias y musicales, resolviendo problemas estructurales de consistencia y escala antes de avanzar hacia la fase de modelado y análisis final.

In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
df_spotify = pd.read_csv('spotify_data clean.csv')
df_poesia = pd.read_csv('poesia_musica_sentimientos.csv', on_bad_lines='skip')

**Limpieza técnica para asegurar el cruce (Merge)**

Se transforman los títulos de las canciones a minúsculas y sin espacios para forzar una coincidencia exacta

In [15]:
df_spotify['track_name_clean'] = df_spotify['track_name'].astype(str).str.strip().str.lower()
df_poesia['cancion_clean'] = df_poesia['cancion_recomendada'].astype(str).str.strip().str.lower()

**Fusión de datasets**

Se unen ambos dataframes mediante un Inner Join en una nueva variable llamada df_completo.

In [16]:
df_completo = pd.merge(df_poesia, df_spotify, left_on='cancion_clean', right_on='track_name_clean', how='inner')
print(f"¡Éxito! Hemos vinculado {df_completo.shape[0]} registros de poesía con datos de audio de Spotify.")

¡Éxito! Hemos vinculado 80 registros de poesía con datos de audio de Spotify.


Esto sirve como el puente técnico de integración de datos (Data Integration). Garantiza que las pequeñas inconsistencias de tipeo manual en los nombres de las canciones no rompan el enlace, logrando que el nuevo dataframe combinado conserve únicamente los registros enriquecidos (con datos de poesía y música a la vez).

**Reducción de cardinalidad y optimización**

Se aplica un diccionario de mapeo para agrupar las emociones dispersas detectadas en el EDA, para consolidarla en una nueva variable (emocion_macro) y se eliminan las columnas clonadas (_clean) con el método .drop().

In [17]:
mapa_emociones = {
    'amor': 'romance_y_pasion', 'pasión': 'romance_y_pasion', 'amor simple': 'romance_y_pasion', 'amorosa': 'romance_y_pasion',
    'resistencia': 'protesta_y_rebeldia', 'rebeldía': 'protesta_y_rebeldia', 'protesta': 'protesta_y_rebeldia', 'justicia': 'protesta_y_rebeldia',
    'asombro': 'contemplacion_y_paz', 'paz': 'contemplacion_y_paz', 'silencio': 'contemplacion_y_paz', 'filosofia': 'contemplacion_y_paz',
    'nostalgia': 'melancolia_y_duelo', 'duelo': 'melancolia_y_duelo', 'tristeza': 'melancolia_y_duelo', 'soledad': 'melancolia_y_duelo'
}



**Mapeo del dataset unificado:**

In [18]:
df_completo['emocion_macro'] = df_completo['emocion_tag'].str.strip().str.lower().map(mapa_emociones).fillna('otras')

**Eliminación de columnas auxiliares que no se necesitan**

In [19]:
df_completo = df_completo.drop(columns=['cancion_clean', 'track_name_clean'], errors='ignore')

print("Mapeo completado con éxito. Conteo de la nueva variable 'emocion_macro':")
print(df_completo['emocion_macro'].value_counts())

Mapeo completado con éxito. Conteo de la nueva variable 'emocion_macro':
emocion_macro
otras                  59
melancolia_y_duelo      9
contemplacion_y_paz     5
protesta_y_rebeldia     5
romance_y_pasion        2
Name: count, dtype: int64


Al ejecutar el mapeo y la optimización de columnas después del merge, se asegura que el flujo de trabajo resuelva primero la integración técnica de las canciones y se neutralice la dispersión de las emociones. Esto garantiza que el dataset final quede 100% limpio, ligero en memoria y listo para el análisis.